# Basic NLP with `text_records()`

This notebook shows a simple handoff from `crategraph` to an NLP library.

The point of using `crategraph` here is that we do not read every file directly. We use the RO-Crate graph to select file entities, annotate those files with genre metadata from related entities, keep provenance and entity metadata with `text_records()`, then compare a couple of genre-specific subgraphs.


## Install TextBlob

Run this cell once if TextBlob is not already available in your notebook environment.

In [ ]:
!uv pip install textblob

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from textblob import TextBlob

from crategraph import Crate

## Load the crate

Load the Australian Corpus of English crate from `data/`.

In [ ]:
crate = Crate(Path("../../data/ldaca/Australian_Corpus_of_English"))
crate.summary()

In [ ]:
crate.glimpse()

## Annotate files with graph-derived genre

Start with file entities, then derive a `genre` property from the file-to-genre relationship recorded in the crate. The annotation is a graph transform, so the derived field behaves like any other entity property in later filtering and record export.


In [ ]:
tagged = crate.annotate_entities(
    genre=lambda entity: entity.related("ldac:linguisticGenre").join("name")
)

text_files = tagged.select(entity_types=["File"])
text_files.summary()

Count the derived genre annotations first, then choose a couple of genre subgraphs to compare. For each selected genre, keep both the graph selection and the text records extracted from that graph.


In [ ]:
genre_counts = Counter()
for record in text_files.entity_records():
    genre = record.get("genre")
    if genre is not None:
        genre_counts[genre] += 1

genre_counts

In [ ]:
genres = ["Report", "Narrative"]

genre_graphs = {}
records_by_genre = {}
for genre in genres:
    graph = text_files.where(genre=genre)
    genre_graphs[genre] = graph
    records_by_genre[genre] = list(graph.text_records(include_properties=["name", "genre"]))

counts = {}
for genre, graph in genre_graphs.items():
    counts[genre] = len(graph.entities)

counts

## Hand selected subgraphs to NLP tools

`text_records()` returns one row-like record per text unit, with provenance columns kept alongside the text. Because `annotate_entities()` wrote `genre` back onto the file entities, `include_properties` carries both native metadata (`name`) and the derived genre into the same rows.


In [ ]:
preview_rows = []
for records in records_by_genre.values():
    preview_rows.extend(records[:3])

pd.DataFrame(preview_rows)[["entity_id", "genre", "name", "text"]]

In [ ]:
word_pattern = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def word_count(text):
    return len(word_pattern.findall(text))


def analyse_record(record):
    analysed = dict(record)
    analysed["word_count"] = word_count(record["text"])
    analysed["polarity"] = TextBlob(record["text"]).sentiment.polarity
    return analysed


analysed_by_genre = {}
for genre, records in records_by_genre.items():
    analysed_records = []
    for record in records:
        analysed_records.append(analyse_record(record))
    analysed_by_genre[genre] = analysed_records

preview = []
for records in analysed_by_genre.values():
    preview.extend(records[:3])

pd.DataFrame(preview)[["entity_id", "genre", "word_count", "polarity", "name"]]

## Compare the genre subgraphs

The grouping column comes from RO-Crate relationships, not from file names. Here the comparison follows the two graph selections directly rather than regrouping the whole corpus table.


In [ ]:
def mean(values):
    values = list(values)
    return sum(values) / len(values) if values else None


def summarise_genre(genre, records):
    word_counts = []
    polarities = []
    for record in records:
        word_counts.append(record["word_count"])
        polarities.append(record["polarity"])

    return {
        "genre": genre,
        "documents": len(records),
        "mean_words": mean(word_counts),
        "mean_polarity": mean(polarities),
    }


summary = []
for genre, records in analysed_by_genre.items():
    summary.append(summarise_genre(genre, records))

pd.DataFrame(summary).set_index("genre")